## Result agregation ##

In [1]:
# Optional: Install dependencies if not already present
INSTALL_DEPS = False

if INSTALL_DEPS:
    import sys
    import subprocess
    
    packages = [
        'numpy',
        'scikit-learn',
        'ucimlrepo',
        'qiskit',
        'qiskit-machine-learning',
        'torch',
        'matplotlib'
    ]
    
    for pkg in packages:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

In [2]:
import os
import torch
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
import torch.nn as nn

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import SparsePauliOp
from qiskit_machine_learning.gradients import ParamShiftEstimatorGradient
from qiskit_machine_learning.connectors import TorchConnector
from qiskit_machine_learning.neural_networks import EstimatorQNN

In [3]:
def ansatz_trimmed_reverse_q0_param_count(n_qubits: int, depth: int) -> int:
    m = depth // 2
    if m == 0:
        return 0
    full = 4 * n_qubits
    last = 3 * n_qubits + 2
    return (m - 1) * full + last

def odra_ansatz(n_qubits: int, depth: int) -> QuantumCircuit:
    n_macro = depth // 2
    theta = ParameterVector("theta", ansatz_trimmed_reverse_q0_param_count(n_qubits, depth))
    qc = QuantumCircuit(n_qubits)
    p = 0

    for j in range(n_macro):
        last_layer = (j == n_macro - 1)

        for i in range(n_qubits):
            qc.ry(theta[p + i], i)
        p += n_qubits

        for i in range(n_qubits):
            control = i
            target = (i + 1) % n_qubits
            qc.rz(theta[p + i], target)
            qc.cz(control, target)
        p += n_qubits

        for i in range(n_qubits):
            qc.rx(theta[p + i], i)
        p += n_qubits

        if last_layer:
            for k in range(2):
                i = k
                control = i
                target = (i - 1) % n_qubits
                qc.ry(theta[p + k], target)
                qc.cz(control, target)
            p += 2
        else:
            for i in range(n_qubits):
                control = i
                target = (i - 1) % n_qubits
                qc.ry(theta[p + i], target)
                qc.cz(control, target)
            p += n_qubits

    assert p == len(theta)
    return qc

def simulator_ansatz(n_qubits: int, depth: int) -> QuantumCircuit:
    n_macro = depth // 2
    theta = ParameterVector("theta", ansatz_trimmed_reverse_q0_param_count(n_qubits, depth))
    qc = QuantumCircuit(n_qubits)
    param_idx = 0

    for j in range(n_macro):
        last_layer = (j == n_macro - 1)

        for i in range(n_qubits):
            qc.ry(theta[param_idx], i)
            param_idx += 1

        for i in range(n_qubits):
            control = i
            target = (i + 1) % n_qubits
            qc.crx(theta[param_idx], control, target)
            param_idx += 1

        for i in range(n_qubits):
            qc.rx(theta[param_idx], i)
            param_idx += 1

        if last_layer:
            for k in range(2):
                i = k
                control = i
                target = (i - 1) % n_qubits
                qc.cry(theta[param_idx], control, target)
                param_idx += 1
        else:
            for i in range(n_qubits):
                control = i
                target = (i - 1) % n_qubits
                qc.cry(theta[param_idx], control, target)
                param_idx += 1

    assert param_idx == len(theta)
    return qc

In [4]:
class HybridModel(nn.Module):
    def __init__(self, ansatz_circuit, num_qubits):
        super().__init__()
        self.feature_map = self._create_angle_encoding(num_qubits)
        
        self.qc = QuantumCircuit(num_qubits)
        self.qc.compose(self.feature_map, qubits=range(num_qubits), inplace=True)
        self.qc.compose(ansatz_circuit, inplace=True)
        
        input_params = list(self.feature_map.parameters)
        weight_params = list(ansatz_circuit.parameters)
        
        observable = SparsePauliOp.from_list([("I" * (num_qubits - 1) + "Z", 1)])
        estimator = StatevectorEstimator()
        gradient = ParamShiftEstimatorGradient(estimator)
        
        self.qnn = EstimatorQNN(
            circuit=self.qc,
            observables=observable,
            input_params=input_params,
            weight_params=weight_params,
            estimator=estimator,
            gradient=gradient
        )
        self.quantum_layer = TorchConnector(self.qnn)
    
    def _create_angle_encoding(self, num_qubits: int) -> QuantumCircuit:
        qc_data = QuantumCircuit(num_qubits)
        input_params = ParameterVector('x', num_qubits)
        for i in range(num_qubits):
            qc_data.ry(input_params[i], i)
        return qc_data
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.quantum_layer(x)

In [7]:
NUM_QUBITS = 5
EPOCH_TO_EVALUATE = 30
MODELS = ['Odra', 'Simulator']
DEPTHS = [2, 4, 6]
CONDITIONS = ['Ideal', 'Noise']
BASE_DIR = "." 

results = []
files_found = 0
files_missing = 0

for depth in DEPTHS:
    for model_name in MODELS:
        for condition in CONDITIONS:
            fold_accs = []
            fold_f1s = []
            
            for fold in range(1, 6):
                # 1. Paths to data and weights
                data_path = os.path.join(BASE_DIR, f"Data/fold_{fold}/test_data.csv")
                
                if model_name == 'Odra':
                    if condition == 'Ideal':
                        weight_file = f"Odra_fold_{fold}_depth_{depth}_epoch_{EPOCH_TO_EVALUATE}_weights.pth"
                    else:
                        weight_file = f"Odra_fold_{fold}_depth_{depth}_noise_epoch_{EPOCH_TO_EVALUATE}_weights.pth"
                else: # Simulator
                    if condition == 'Ideal':
                        weight_file = f"Simulator_fold_{fold}_depth_{depth}_epoch_{EPOCH_TO_EVALUATE}_weights.pth"
                    else:
                        weight_file = f"Simulator_fold_{fold}_depth_{depth}_noise_epoch_{EPOCH_TO_EVALUATE}_weights.pth"
                
                weight_path = os.path.join(BASE_DIR, f"Models/Weights/depth {depth}/{model_name}/fold_{fold}/{weight_file}")
                
                if model_name == 'Simulator' and condition == 'Ideal' and not os.path.exists(weight_path):
                    alt_weight_file = f"Simulator_fold_{fold}_depth_{depth}_epoch_{EPOCH_TO_EVALUATE}_weights.pth"
                    alt_weight_path = os.path.join(BASE_DIR, f"Models/Weights/depth {depth}/{model_name}/fold_{fold}/{alt_weight_file}")
                    if os.path.exists(alt_weight_path):
                        weight_path = alt_weight_path
                
                # Verify if files exist
                if not os.path.exists(data_path):
                    print(f"MISSING DATA: {data_path}")
                    files_missing += 1
                    continue
                    
                if not os.path.exists(weight_path):
                    print(f"MISSING WEIGHTS: {weight_path}")
                    files_missing += 1
                    continue
                
                files_found += 1
                
                # 2. Load data
                df = pd.read_csv(data_path)
                X_test = torch.tensor(df.drop('target', axis=1).values, dtype=torch.float32)
                y_test = df['target'].values
                
                # 3. Load corresponding model
                if model_name == 'Odra':
                    ansatz = odra_ansatz(NUM_QUBITS, depth)
                else:
                    ansatz = simulator_ansatz(NUM_QUBITS, depth)
                    
                model = HybridModel(ansatz, NUM_QUBITS)
                model.load_state_dict(torch.load(weight_path, weights_only=True))
                model.eval()
                
                # 4. Predictions and metrics
                with torch.no_grad():
                    preds = (torch.round(model(X_test), decimals=5) > 0).float() * 2 - 1
                    preds_np = preds.numpy().flatten()
                    
                    acc = accuracy_score(y_test, preds_np)
                    f1 = f1_score(y_test, preds_np, pos_label=1)
                    
                    fold_accs.append(acc)
                    fold_f1s.append(f1)
            
            # 5. Aggregation (if folds were found for given parameters)
            if len(fold_accs) > 0:
                mean_acc = np.mean(fold_accs)
                std_acc = np.std(fold_accs)
                mean_f1 = np.mean(fold_f1s)
                std_f1 = np.std(fold_f1s)
                
                results.append({
                    'Depth': depth,
                    'Model': model_name,
                    'Noise/Ideal': condition,
                    'Mean acc +-std': f"{mean_acc:.4f} ± {std_acc:.4f}",
                    'Mean F1 +-std': f"{mean_f1:.4f} ± {std_f1:.4f}"
                })

print(f"\nFound valid weight files: {files_found}")
print(f"Missing weight/data files: {files_missing}\n")

# Create final Pandas table
df_results = pd.DataFrame(results)

# Display Markdown table
if not df_results.empty:
    from IPython.display import display, Markdown
    display(Markdown(df_results.to_markdown(index=False)))
else:
    print("Table is empty because no models were matched. Check the logs above.")

MISSING WEIGHTS: ./Models/Weights/depth 6/Simulator/fold_1/Simulator_fold_1_depth_6_noise_epoch_30_weights.pth
MISSING WEIGHTS: ./Models/Weights/depth 6/Simulator/fold_2/Simulator_fold_2_depth_6_noise_epoch_30_weights.pth
MISSING WEIGHTS: ./Models/Weights/depth 6/Simulator/fold_3/Simulator_fold_3_depth_6_noise_epoch_30_weights.pth
MISSING WEIGHTS: ./Models/Weights/depth 6/Simulator/fold_4/Simulator_fold_4_depth_6_noise_epoch_30_weights.pth
MISSING WEIGHTS: ./Models/Weights/depth 6/Simulator/fold_5/Simulator_fold_5_depth_6_noise_epoch_30_weights.pth

Found valid weight files: 55
Missing weight/data files: 5



|   Depth | Model     | Noise/Ideal   | Mean acc +-std   | Mean F1 +-std   |
|--------:|:----------|:--------------|:-----------------|:----------------|
|       2 | Odra      | Ideal         | 0.8696 ± 0.0180  | 0.8422 ± 0.0262 |
|       2 | Odra      | Noise         | 0.8688 ± 0.0203  | 0.8395 ± 0.0304 |
|       2 | Simulator | Ideal         | 0.8958 ± 0.0151  | 0.8745 ± 0.0239 |
|       2 | Simulator | Noise         | 0.8878 ± 0.0173  | 0.8636 ± 0.0273 |
|       4 | Odra      | Ideal         | 0.8761 ± 0.0280  | 0.8446 ± 0.0447 |
|       4 | Odra      | Noise         | 0.8659 ± 0.0228  | 0.8310 ± 0.0389 |
|       4 | Simulator | Ideal         | 0.8761 ± 0.0258  | 0.8458 ± 0.0436 |
|       4 | Simulator | Noise         | 0.8666 ± 0.0252  | 0.8319 ± 0.0423 |
|       6 | Odra      | Ideal         | 0.8827 ± 0.0276  | 0.8519 ± 0.0453 |
|       6 | Odra      | Noise         | 0.8557 ± 0.0367  | 0.8095 ± 0.0614 |
|       6 | Simulator | Ideal         | 0.8776 ± 0.0262  | 0.8455 ± 0.0427 |